In [0]:
import pandas as pd
import io
import requests

In [0]:
url = "https://datasets.imdbws.com/name.basics.tsv.gz"
name_basics_df = pd.read_csv(url, compression='gzip', sep='\t')

name_basics_df.head()

,nconst,primaryName,birthYear,deathYear,primaryProfession,knownForTitles
0,nm0000001,Fred Astaire,1899,1987,"actor,miscellaneous,producer","tt0072308,tt0050419,tt0027125,tt0031983"
1,nm0000002,Lauren Bacall,1924,2014,"actress,soundtrack,archive_footage","tt0037382,tt0075213,tt0117057,tt0038355"
2,nm0000003,Brigitte Bardot,1934,\N,"actress,music_department,producer","tt0057345,tt0049189,tt0056404,tt0054452"
3,nm0000004,John Belushi,1949,1982,"actor,writer,music_department","tt0072562,tt0077975,tt0080455,tt0078723"
4,nm0000005,Ingmar Bergman,1918,2007,"writer,director,actor","tt0050986,tt0069467,tt0050976,tt0083922"


In [0]:
url = "https://datasets.imdbws.com/title.basics.tsv.gz"
title_basics_df = pd.read_csv(url, compression='gzip', sep='\t')

title_basics_df.head()

<command-3511027810573584>:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  title_basics_df = pd.read_csv(url, compression='gzip', sep='\t')


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,\N,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,\N,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,\N,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,\N,1,Short


In [0]:

%sql SHOW DATABASES;


databaseName
bronze
default


In [0]:
%sql
-- Drop the database using IF EXISTS
-- DROP DATABASE CASCADE - deleta um database e suas tabelas 
DROP DATABASE IF EXISTS bronze CASCADE;

In [0]:
%sql CREATE DATABASE bronze;

In [0]:
%sql SHOW DATABASES;

databaseName
bronze
default


In [0]:
name_basics_spark_df = spark.createDataFrame(name_basics_df)
name_basics_spark_df.write.mode("overwrite").saveAsTable("bronze.name_basics")

In [0]:
title_basics_spark_df = spark.createDataFrame(title_basics_df)
title_basics_spark_df.write.mode("overwrite").saveAsTable("bronze.title_basics")

/databricks/spark/python/pyspark/sql/pandas/conversion.py:467: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Could not convert '0' with type str: tried to convert to int64
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


In [0]:
%sql SELECT * FROM bronze.name_basics LIMIT 10

nconst,primaryName,birthYear,deathYear,primaryProfession,knownForTitles
nm0000001,Fred Astaire,1899,1987,"actor,miscellaneous,producer","tt0072308,tt0050419,tt0027125,tt0031983"
nm0000002,Lauren Bacall,1924,2014,"actress,soundtrack,archive_footage","tt0037382,tt0075213,tt0117057,tt0038355"
nm0000003,Brigitte Bardot,1934,\N,"actress,music_department,producer","tt0057345,tt0049189,tt0056404,tt0054452"
nm0000004,John Belushi,1949,1982,"actor,writer,music_department","tt0072562,tt0077975,tt0080455,tt0078723"
nm0000005,Ingmar Bergman,1918,2007,"writer,director,actor","tt0050986,tt0069467,tt0050976,tt0083922"
nm0000006,Ingrid Bergman,1915,1982,"actress,producer,soundtrack","tt0034583,tt0038109,tt0036855,tt0038787"
nm0000007,Humphrey Bogart,1899,1957,"actor,producer,miscellaneous","tt0034583,tt0043265,tt0037382,tt0033870"
nm0000008,Marlon Brando,1924,2004,"actor,director,writer","tt0078788,tt0068646,tt0047296,tt0070849"
nm0000009,Richard Burton,1925,1984,"actor,producer,director","tt0061184,tt0087803,tt0059749,tt0057877"
nm0000010,James Cagney,1899,1986,"actor,director,producer","tt0029870,tt0031867,tt0042041,tt0034236"


In [0]:
%sql SELECT * FROM bronze.title_basics LIMIT 10

tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
tt0000001,short,Carmencita,Carmencita,0,1894,\N,1,"Documentary,Short"
tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,\N,5,"Animation,Short"
tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance"
tt0000004,short,Un bon bock,Un bon bock,0,1892,\N,12,"Animation,Short"
tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,\N,1,Short
tt0000006,short,Chinese Opium Den,Chinese Opium Den,0,1894,\N,1,Short
tt0000007,short,Corbett and Courtney Before the Kinetograph,Corbett and Courtney Before the Kinetograph,0,1894,\N,1,"Short,Sport"
tt0000008,short,Edison Kinetoscopic Record of a Sneeze,Edison Kinetoscopic Record of a Sneeze,0,1894,\N,1,"Documentary,Short"
tt0000009,movie,Miss Jerry,Miss Jerry,0,1894,\N,45,Romance
tt0000010,short,Leaving the Factory,La sortie de l'usine Lumière à Lyon,0,1895,\N,1,"Documentary,Short"


In [0]:
%sql DROP DATABASE IF EXISTS silver CASCADE;

In [0]:
%sql CREATE DATABASE silver

In [0]:
from pyspark.sql.functions import explode
from pyspark.sql.functions import split

name_basics_titles_spark_df = name_basics_spark_df.select(name_basics_spark_df.nconst, name_basics_spark_df.primaryName, explode(split(name_basics_spark_df.knownForTitles, ',')).alias("tconst"))
name_basics_titles_spark_df.show(10)

+---------+---------------+---------+
|   nconst|    primaryName|   tconst|
+---------+---------------+---------+
|nm0000001|   Fred Astaire|tt0072308|
|nm0000001|   Fred Astaire|tt0050419|
|nm0000001|   Fred Astaire|tt0027125|
|nm0000001|   Fred Astaire|tt0031983|
|nm0000002|  Lauren Bacall|tt0037382|
|nm0000002|  Lauren Bacall|tt0075213|
|nm0000002|  Lauren Bacall|tt0117057|
|nm0000002|  Lauren Bacall|tt0038355|
|nm0000003|Brigitte Bardot|tt0057345|
|nm0000003|Brigitte Bardot|tt0049189|
+---------+---------------+---------+
only showing top 10 rows



In [0]:
name_basics_titles_spark_df.write.mode("overwrite").saveAsTable("silver.name_basics_titles")

In [0]:
%sql SELECT * FROM silver.name_basics_titles LIMIT 100

nconst,primaryName,tconst
nm0000001,Fred Astaire,tt0072308
nm0000001,Fred Astaire,tt0050419
nm0000001,Fred Astaire,tt0027125
nm0000001,Fred Astaire,tt0031983
nm0000002,Lauren Bacall,tt0037382
nm0000002,Lauren Bacall,tt0075213
nm0000002,Lauren Bacall,tt0117057
nm0000002,Lauren Bacall,tt0038355
nm0000003,Brigitte Bardot,tt0057345
nm0000003,Brigitte Bardot,tt0049189


In [0]:
%sql SELECT isAdult, COUNT(*)
FROM bronze.title_basics
--WHERE isAdult = 0 OR isAdult = 1
GROUP BY isAdult 

isAdult,count(1)
1987,13
2016,20
2012,1
2020,9
1958,1
1972,29
1988,5
2019,7
2017,17
1977,20


In [0]:
%sql SELECT tconst, titleType, originalTitle, startYear as year,
CASE 
  WHEN isAdult = 1 THEN TRUE
  WHEN isAdult = 0 THEN FALSE
END AS isAdult
FROM bronze.title_basics
WHERE isAdult = 0 OR isAdult = 1

tconst,titleType,originalTitle,year,isAdult
tt30096209,tvEpisode,What You Need to Know Before You Stay at Disney's Port Orleans Resort - French Quarter,2023,false
tt30096212,movie,Devi Danger,2024,false
tt30096213,tvSpecial,Red Boxing Promotions Presents: Desert Storm,2023,false
tt30096215,short,Sreekaram - The Beginning,2023,false
tt30096218,tvEpisode,Just Relaxing,2023,true
tt30096219,video,Fake Smile,2009,false
tt3009622,movie,Reuber,2013,false
tt30096221,movie,Ella McCay,2025,false
tt30096222,tvEpisode,Los Diez Mandamientos,2022,false
tt30096228,tvEpisode,Tricia Penrose v Dean Sullivan,2023,false


In [0]:
%sql CREATE TABLE silver.title_basics
AS SELECT tconst, titleType, originalTitle, startYear as year,
CASE 
  WHEN isAdult = 1 THEN TRUE
  WHEN isAdult = 0 THEN FALSE
END AS isAdult
FROM bronze.title_basics
WHERE isAdult = 0 OR isAdult = 1

num_affected_rows,num_inserted_rows


In [0]:
%sql SELECT * FROM silver.title_basics
LIMIT 100

tconst,titleType,originalTitle,year,isAdult
tt0000001,short,Carmencita,1894,false
tt0000002,short,Le clown et ses chiens,1892,false
tt0000003,short,Pauvre Pierrot,1892,false
tt0000004,short,Un bon bock,1892,false
tt0000005,short,Blacksmith Scene,1893,false
tt0000006,short,Chinese Opium Den,1894,false
tt0000007,short,Corbett and Courtney Before the Kinetograph,1894,false
tt0000008,short,Edison Kinetoscopic Record of a Sneeze,1894,false
tt0000009,movie,Miss Jerry,1894,false
tt0000010,short,La sortie de l'usine Lumière à Lyon,1895,false


In [0]:
%sql DROP DATABASE IF EXISTS gold CASCADE

In [0]:
%sql CREATE DATABASE gold

In [0]:
%sql SELECT * 
FROM silver.title_basics AS T,
  silver.name_basics_titles AS N
WHERE N.tconst = T.tconst
LIMIT 10

tconst,titleType,originalTitle,year,isAdult,nconst,primaryName,tconst
tt0000015,short,Autour d'une cabine,1894,false,nm1335271,Gaston Paulin,tt0000015
tt0000174,short,Výstavní párkar a lepic plakátù,1898,false,nm0471818,Jan Krízenecký,tt0000174
tt0000174,short,Výstavní párkar a lepic plakátù,1898,false,nm1024447,Ferdinand Gýra,tt0000174
tt0000305,short,L'Habanera,1900,false,nm0112631,Valentine Brouat,tt0000305
tt0000451,short,Mary Jane's Mishap,1903,false,nm0809419,Laura Bayley,tt0000451
tt0000621,short,That Fatal Sneeze,1907,false,nm0693275,Gertie Potter,tt0000621
tt0000621,short,That Fatal Sneeze,1907,false,nm2140812,Thurston Harris,tt0000621
tt0000748,short,The Redman and the Child,1908,false,nm0537238,Johnny Mahr,tt0000748
tt0000748,short,The Redman and the Child,1908,false,nm0409390,Charles Inslee,tt0000748
tt0000862,movie,Faldgruben,1909,false,nm0878467,Emanuel Tvede,tt0000862


In [0]:
%sql CREATE TABLE gold.names_titles AS
SELECT N.nconst, N.primaryName, T.*
FROM silver.title_basics AS T,
  silver.name_basics_titles AS N
WHERE N.tconst = T.tconst

num_affected_rows,num_inserted_rows


In [0]:
%sql SELECT * FROM gold.names_titles
ORDER BY originalTitle, primaryName
LIMIT 1000

nconst,primaryName,tconst,titleType,originalTitle,year,isAdult
nm2506382,Stephen Sung,tt2386381,tvSeries,!Next?,1994,false
nm5794862,Aaron Margolis-Greenbaum,tt21091602,tvSeries,#,2022,false
nm11054261,Aaron Maurice,tt21091602,tvSeries,#,2022,false
nm12571512,Adrienne Schmucker,tt21091602,tvSeries,#,2022,false
nm15269991,Aftin Brown,tt21091602,tvSeries,#,2022,false
nm14012320,Alyson Cline,tt21091602,tvSeries,#,2022,false
nm12646409,Alyssa Baskins,tt21091602,tvSeries,#,2022,false
nm11091998,Amelia Morris,tt21091602,tvSeries,#,2022,false
nm13827163,Amity Aschliman,tt21091602,tvSeries,#,2022,false
nm15262189,Anastasia Hinton,tt21091602,tvSeries,#,2022,false


In [0]:
%sql SELECT * FROM gold.names_titles
WHERE primaryName like "%Kevin Bacon%"
LIMIT 1000

nconst,primaryName,tconst,titleType,originalTitle,year,isAdult
nm8407192,Kevin Bacon Hervieux,tt6026054,tvMovie,Innu Nikamu: La Grande Tradition,2017,false
nm8407192,Kevin Bacon Hervieux,tt9207434,movie,Innu Nikamu: chanter la résistance,2017,false
nm0000102,Kevin Bacon,tt0164052,movie,Hollow Man,2000,false
nm0000102,Kevin Bacon,tt0327056,movie,Mystic River,2003,false
nm4025714,Kevin Bacon,tt8429394,movie,The Return,2020,false
nm4025714,Kevin Bacon,tt0098193,movie,The Return of Swamp Thing,1989,false
nm4025714,Kevin Bacon,tt1737781,video,The Suit,2010,false
nm3636162,Kevin Bacon,tt2063666,movie,Hello I Must Be Going,2012,false
nm4025714,Kevin Bacon,tt3067274,movie,The Editor,2014,false
nm9323132,Kevin Bacon,tt0259795,tvSeries,The Old Grey Whistle Test,1971,false


In [0]:
%sql SELECT B.primaryName, COUNT(B.originalTitle) AS Quant_filmes
FROM gold.names_titles AS B
GROUP BY B.primaryName
ORDER BY Quant_filmes DESC

primaryName,Quant_filmes
David Smith,596
Michael Smith,555
Chris Smith,513
Mike Smith,504
Alex,499
Michael Johnson,498
David Williams,491
Chris Jones,484
Chris Johnson,482
David Brown,476


In [0]:
%sql SELECT B.primaryName, COUNT(*)
FROM gold.names_titles AS A,
  gold.names_titles AS B
WHERE A.primaryName = "Kevin Bacon"
AND B.primaryName <> "Kevin Bacon"
AND A.tconst = B.tconst
GROUP BY B.primaryName
ORDER BY COUNT(*) DESC

primaryName,count(1)
Mik Glaisher,4
Stephen Fellows,4
Andy Peake,4
Jim Scharf,3
Bill Jones,2
Mike O'Neill,2
Corey Rich,2
Robert Bluemke,2
Gary Cobb,2
Tim Scott,2


In [0]:
%sql SELECT B.primaryName, COUNT(B.originalTitle) AS Quant_filmes
FROM gold.names_titles AS B
GROUP BY B.primaryName
ORDER BY Quant_filmes DESC
LIMIT 3

primaryName,Quant_filmes
David Smith,596
Michael Smith,555
Chris Smith,513


In [0]:
%sql SELECT B.primaryName, COUNT(B.originalTitle) AS Quant_filmes
FROM gold.names_titles AS B
GROUP BY B.primaryName
ORDER BY Quant_filmes ASC
LIMIT 1

primaryName,Quant_filmes
Rolf Knudssøn,1
